<a href="https://colab.research.google.com/github/jinsujini/SSWU_AI_TEAM4/blob/main/newmindnewstart.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 경건하게 새마음 새출발 진수 코드

In [ ]:
!cp /content/drive/MyDrive/Hip\ Circles_Swimming.zip /content/


In [ ]:
!unzip /content/Hip\ Circles_Swimming.zip -d /content/data/


스트리밍 출력 내용이 길어서 마지막 5000줄이 삭제되었습니다.
  inflating: /content/data/Hip Circles/actorP116(1)_2/frame_000146_skeleton.png  
  inflating: /content/data/Hip Circles/actorP116(1)_2/frame_000147_skeleton.png  
  inflating: /content/data/Hip Circles/actorP116(1)_2/frame_000148_skeleton.png  
  inflating: /content/data/Hip Circles/actorP116(1)_2/frame_000149_skeleton.png  
  inflating: /content/data/Hip Circles/actorP116(1)_2/frame_000150_skeleton.png  
  inflating: /content/data/Hip Circles/actorP116(1)_2/frame_000151_skeleton.png  
  inflating: /content/data/Hip Circles/actorP116(1)_2/frame_000152_skeleton.png  
  inflating: /content/data/Hip Circles/actorP116(1)_2/frame_000153_skeleton.png  
  inflating: /content/data/Hip Circles/actorP116(1)_2/frame_000154_skeleton.png  
  inflating: /content/data/Hip Circles/actorP116(1)_2/frame_000155_skeleton.png  
  inflating: /content/data/Hip Circles/actorP116(1)_2/frame_000156_skeleton.png  
  inflating: /content/data/Hip Circles/actorP116(1)_2/frame_00

# 데이터


In [ ]:
video_list = sorted(os.listdir(action_path))
print(f"🎞️ 총 폴더 수: {len(video_list)}")

seq_idx = 0

for video_folder in tqdm(video_list):
    video_path = os.path.join(action_path, video_folder)
    if not os.path.isdir(video_path):
        continue

    # 🎯 wrong 데이터
    if 'wrong' in video_folder:
        img_dir = os.path.join(video_path, 'skeleton_images')
        label_path = os.path.join(video_path, 'labels.json')
        image_paths = sorted(glob.glob(os.path.join(img_dir, '*.png')))
        label_info = []
        if os.path.exists(label_path):
            with open(label_path, 'r') as f:
                label_info = json.load(f)
    else:
        image_paths = sorted(glob.glob(os.path.join(video_path, '*.png')))
        label_info = None

    if len(image_paths) > max_frames:
        image_paths = image_paths[:max_frames]

    total_frames = len(image_paths)
    if total_frames < seq_len:
        continue

    for start in range(0, total_frames - seq_len + 1, stride):
        end = start + seq_len
        frame_paths = image_paths[start:end]

        imgs = []
        for path in frame_paths:
            try:
                img = Image.open(path).convert('RGB')
                img = base_transform(img)
                imgs.append(img)
            except:
                break

        if len(imgs) < seq_len:
            continue

        images_tensor = torch.stack(imgs)

        # 🎯 정오 판단
        if label_info:
            frame_labels = [item.get('is_correct', 1) for item in label_info[start:end] if item]
            label = 0 if any(l == 0 for l in frame_labels) else 1

            joint_vector = torch.zeros(len(joint_names))
            for item in label_info[start:end]:
                for jname in item.get("wrong_joints", []):
                    if jname in joint_names:
                        joint_vector[joint_names.index(jname)] = 1
        else:
            label = 1
            joint_vector = torch.zeros(len(joint_names))

        # 💾 저장
        save_path = os.path.join(save_dir, f'seq_{seq_idx:05d}_{video_folder}.pt')
        torch.save({
            'frames': images_tensor,
            'label': label,
            'joint_vector': joint_vector
        }, save_path)

        seq_idx += 1

print(f"\n✅ 저장 완료！총 {seq_idx}개 시퀀스")


🎞️ 총 폴더 수: 280


100%|██████████| 280/280 [49:35<00:00, 10.63s/it]


✅ 저장 완료！총 639개 시퀀스


In [5]:
import os
import torch
from collections import Counter

pt_dir = '/content/drive/MyDrive/preprocessed_data/Swimming_pt'

all_data = []

for fname in sorted(os.listdir(pt_dir)):
    if fname.endswith('.pt'):
        fpath = os.path.join(pt_dir, fname)
        try:
            data = torch.load(fpath)
            all_data.append(data)
        except Exception as e:
            print(f"❌ {fname} 로딩 실패 → {e}")

# label 비율 확인
labels = [int(d['label']) for d in all_data]
print(f"🧮 총 시퀀스 수: {len(all_data)}")
print("📊 정오 분포:", Counter(labels))


🧮 총 시퀀스 수: 639
📊 정오 분포: Counter({1: 403, 0: 236})


# 학습


In [76]:
# 기본 라이브러리
import os
import json
import glob
import random
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt

# Torch 관련
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import torch.nn.functional as F
from torchvision import transforms
from torch.cuda.amp import GradScaler, autocast

# Colab: 구글 드라이브 마운트
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [77]:
from torchvision import transforms

# 평균·표준편차는 ImageNet 기준
base_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])


In [78]:
from sklearn.model_selection import train_test_split
import glob

pt_dir = '/content/drive/MyDrive/preprocessed_data/Swimming_pt'  # 저장한 pt 디렉토리
all_files = sorted(glob.glob(os.path.join(pt_dir, "*.pt")))
print(f"🗂️ 총 .pt 시퀀스 수: {len(all_files)}")
labels = [d['label'] for d in all_data]  # 👈 all_data와 정확히 1:1
train_data, val_data = train_test_split(
    all_data, test_size=0.2, random_state=42,
    stratify=labels
)


🗂️ 총 .pt 시퀀스 수: 639


In [79]:
from torch.utils.data import Dataset
import torch
import os
import glob

class PreprocessedDataset(Dataset):
    def __init__(self, data_dir, file_list=None):
        self.file_paths = sorted(glob.glob(os.path.join(data_dir, '*.pt')))
        if file_list:
            self.file_paths = [f for f in self.file_paths if os.path.basename(f) in file_list]

    def __len__(self):
        return len(self.file_paths)

    def __getitem__(self, idx):
        item = torch.load(self.file_paths[idx])
        return item["images"], torch.tensor(item["label"], dtype=torch.float32), item["joints"]


In [80]:
import os, glob, json
import torch
from PIL import Image
from torchvision import transforms
from tqdm import tqdm

# 🧭 경로
dataset_path = '/content/data'  # 이미지 압축 해제 후 경로
action_name = 'Swimming'
action_path = os.path.join(dataset_path, action_name)
save_dir = f'/content/drive/MyDrive/preprocessed_data/{action_name}_pt'
os.makedirs(save_dir, exist_ok=True)

# ⚙️ 파라미터
seq_len = 60
stride = 30
max_frames = 300  # 긴 영상 자르기

# 📐 transform
base_transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor()
])

# ✅ 관절 이름
joint_names = [
    "Head", "Neck", "LShoulder", "LElbow", "LWrist",
    "RShoulder", "RElbow", "RWrist", "Hip", "LKnee", "RKnee", "Ankle"
]


In [81]:
from torch.utils.data import Dataset
import torch
import os

class PreprocessedDataset(Dataset):
    def __init__(self, pt_file_list):
        self.file_list = pt_file_list

    def __len__(self):
        return len(self.file_list)

    def __getitem__(self, idx):
        item = torch.load(self.file_list[idx])
        return item['frames'], torch.tensor(item['label'], dtype=torch.float32), item['joint_vector']


In [82]:
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader

# stratify 기준: 라벨
train_data, val_data = train_test_split(
    all_data, test_size=0.2, random_state=42,
    stratify=labels
)

class PreprocessedSequenceDataset(Dataset):
    def __init__(self, data_list):
        self.data = data_list

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        d = self.data[idx]
        return d['frames'], torch.tensor(d['label'], dtype=torch.float32), d['joint_vector']

# Dataset, DataLoader
train_dataset = PreprocessedSequenceDataset(train_data)
val_dataset = PreprocessedSequenceDataset(val_data)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False, num_workers=2)


In [83]:
import torch.nn as nn
from torch.cuda.amp import GradScaler, autocast

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
class CNNLSTMClassifier(nn.Module):
    def __init__(self, cnn_out_dim=128, hidden_dim=128, num_layers=1, num_joints=12, dropout_prob=0.3):  # ← 요기!
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, cnn_out_dim, 3, padding=1), nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1))
        )
        self.lstm = nn.LSTM(cnn_out_dim, hidden_dim, num_layers, batch_first=True)
        self.dropout = nn.Dropout(p=dropout_prob)  # ← 요기도!
        self.correct_fc = nn.Linear(hidden_dim, 1)
        self.joint_fc   = nn.Linear(hidden_dim, num_joints)

    def forward(self, x):
        b, t, c, h, w = x.size()
        x = x.view(b * t, c, h, w)
        x = self.cnn(x).view(b, t, -1)
        x, _ = self.lstm(x)
        x = self.dropout(x[:, -1])
        return self.correct_fc(x), self.joint_fc(x)

# 🧠 손실 함수
loss_correct = nn.BCEWithLogitsLoss()
loss_joint   = nn.BCEWithLogitsLoss()
model = CNNLSTMClassifier(dropout_prob=0.5).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=5e-5)
scaler = GradScaler()


<ipython-input-83-22e144e91b26>:33: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()
/usr/local/lib/python3.11/dist-packages/torch/amp/grad_scaler.py:132: UserWarning: torch.cuda.amp.GradScaler is enabled, but CUDA is not available.  Disabling.
  warnings.warn(


In [84]:
from tqdm import tqdm

def train_model(model, train_loader, val_loader, epochs=10, lambda_joint=1.0):
    print("🚀 학습 시작 ∼！\n")
    for epoch in range(epochs):
        print(f"🔁 Epoch {epoch+1}/{epochs}")
        model.train()
        train_loss = 0

        for x, y_correct, y_joint in tqdm(train_loader, desc=f"Train {epoch+1}"):
            x = x.to(device)
            y_correct = y_correct.float().view(-1, 1).to(device)
            y_joint = y_joint.float().to(device)

            optimizer.zero_grad()
            with autocast():
                pred_correct, pred_joint = model(x)
                loss1 = loss_correct(pred_correct, y_correct)
                loss2 = loss_joint(pred_joint, y_joint)
                loss = loss1 + lambda_joint * loss2
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            train_loss += loss.item()

        print(f"🧮 Train Loss: {train_loss:.4f}")

        model.eval()
        val_loss = 0
        correct_total = 0
        total = 0
        with torch.no_grad():
            for x, y_correct, y_joint in tqdm(val_loader, desc=f"Val {epoch+1}"):
                x = x.to(device)
                y_correct = y_correct.float().view(-1, 1).to(device)
                y_joint = y_joint.float().to(device)

                pred_correct, pred_joint = model(x)
                loss1 = loss_correct(pred_correct, y_correct)
                loss2 = loss_joint(pred_joint, y_joint)
                val_loss += (loss1 + lambda_joint * loss2).item()

                pred_labels = (pred_correct > 0.5).float()
                correct_total += (pred_labels == y_correct).sum().item()
                total += y_correct.size(0)

        val_acc = 100. * correct_total / total
        print(f"✅ Val Loss: {val_loss:.4f} | Val Accuracy: {val_acc:.2f}%\n")


In [ ]:
# 🔥 학습 실행
train_model(
    model,
    train_loader,
    val_loader,
    epochs=10,          # ← 에폭 수 조절 가능
    lambda_joint=0.7    # ← joint loss 비중 (0.5 ~ 2.0 추천)
)


🚀 학습 시작 ∼！

🔁 Epoch 1/10


Train 1:   0%|          | 0/64 [00:00<?, ?it/s]<ipython-input-84-e8306da480ff>:16: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
/usr/local/lib/python3.11/dist-packages/torch/amp/autocast_mode.py:266: UserWarning: User provided device_type of 'cuda', but CUDA is not available. Disabling
  warnings.warn(
Train 1: 100%|██████████| 64/64 [05:48<00:00,  5.45s/it]


🧮 Train Loss: 73.4343


Val 1: 100%|██████████| 16/16 [00:40<00:00,  2.50s/it]


✅ Val Loss: 17.9911 | Val Accuracy: 36.72%

🔁 Epoch 2/10


...

Train 4: 100%|██████████| 64/64 [05:51<00:00,  5.49s/it]


🧮 Train Loss: 59.9265


Val 4: 100%|██████████| 16/16 [00:45<00:00,  2.83s/it]


✅ Val Loss: 13.8456 | Val Accuracy: 63.28%

🔁 Epoch 5/10


Train 5:  41%|████      | 26/64 [02:21<03:21,  5.30s/it]

In [ ]:
# 추론 함수
def infer_and_decode(model, data_loader, threshold=0.5):
    model.eval()
    all_results = []

    with torch.no_grad():
        for x, _, _ in tqdm(data_loader, desc="🔍 Inference"):
            x = x.to(device)
            is_correct_probs, joint_probs = model(x)
            results = decode_output(is_correct_probs, joint_probs, threshold)
            all_results.extend(results)

    return all_results

# 추론 실행
results = infer_and_decode(model, val_loader)

# 예시 출력
for i, res in enumerate(results[:5]):
    print(f"[Sample {i}] 정오 확률: {res['is_correct_prob']}, 오류 관절: {res['wrong_joints']}")
